In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

silver_hta = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/fact_hta/")
silver_diag = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/fact_diag")
silver_insu = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/fact_insumos")
silver_medica = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/fact_medicamentos")
silver_proced = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/fact_procedimientos")
lkp_nam_diagnosticos = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/diagnóstico/")
silver_dim_insum = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/insumos/")
silver_dim_med = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/medicamentos/")
lkp_nam_procedimiento = spark.read.format("delta").load("/Volumes/workspace/hta_sis/bronze/procedimientos/")

## Transformación para silver_hta (nuestro fact table)

In [0]:
silver_hta = silver_hta.withColumn(
    "SEXO",
    F.when(F.col("SEXO").contains("FEMENINO"), "F").otherwise("M")
)
silver_hta = silver_hta.drop("FECHA_CORTE")
silver_hta = silver_hta.filter(F.col("DEPARTAMENTO") == "LIMA")

In [0]:
silver_hta = silver_hta.filter(F.lower(F.col("DISTRITO")) == "san juan de miraflores")

In [0]:
silver_hta = (
    silver_hta
    .withColumn(
        "DIAS_HOSP",
        F.when(F.col("DIAS_HOSP").isNull(), 0).otherwise(F.col("DIAS_HOSP"))
    )
    .withColumn(
        "PRES_ART_SISTOLICA",
        F.when(F.col("PRES_ART_SISTOLICA").isNull(), 0).otherwise(F.col("PRES_ART_SISTOLICA"))
    )
    .withColumn(
        "PRES_ART_DIASTOLICA",
        F.when(F.col("PRES_ART_DIASTOLICA").isNull(), 0).otherwise(F.col("PRES_ART_DIASTOLICA"))
    )
    .withColumn(
        "PERIODO_ATENCION",
        F.col("FECHA_ATENCION")
    )
    .withColumn(
        "FECHA_ATENCION",
        F.to_date(F.col("FECHA_ATENCION"), "yyyyMMdd")
    )
    .withColumn(
        "FECATE_POST_FECFED",
        F.when(F.col("FECATE_POST_FECFED")=='SI', F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "ES_CAPITA",
        F.when(F.col("ES_CAPITA")=='S',F.lit(1)).otherwise(F.lit(0))
    )
)

In [0]:
silver_hta = silver_hta.drop("DEPARTAMENTO","PROVINCIA","DISTRITO","UBIGEO","ULTIMO_MES_CONSUMOS")

## Transformación para silver_diag

In [0]:
silver_diag = silver_diag.withColumn(
    "TIPO_DIAGNOSTICO",
    F.when(F.col("TIPO_DIAGNOSTICO").contains("DEFINITIVO"), "D")
    .otherwise(
        F.when(F.col("TIPO_DIAGNOSTICO").contains("PRESUNTIVO"), "P")
        .otherwise("R")
    )
)

In [0]:
silver_diag = silver_diag.drop(F.col("FECHA_CORTE"), F.col("ULTIMO_MES_CONSUMOS"))

## Pequeño cambio de nombre a un catalogo de diagonosticos de los pacientes, aparte de padecer hipertensión arterial

In [0]:
lkp_nam_diagnosticos = lkp_nam_diagnosticos.withColumnRenamed("c10_nombre","nombre_diagnostico")

In [0]:
silver_dim_med = silver_dim_med.withColumn(
    "presentacion",F.when(F.col("presentacion").isNull(),"N/A").otherwise(F.col("presentacion"))
 )

In [0]:
# tables = {
#     "hta_sis.silver_hta": silver_hta,
#     "hta_sis.silver_diag": silver_diag,
#     "hta_sis.silver_insu": silver_insu,
#     "hta_sis.silver_medica": silver_medica,
#     "hta_sis.silver_proced": silver_proced,
#     "hta_sis.silver_nam_diag": lkp_nam_diagnosticos,
#     "hta_sis.silver_dim_insum": silver_dim_insum,
#     "hta_sis.silver_dim_med": silver_dim_med,
#     "hta_sis.silver_dim_proc": lkp_nam_procedimiento
# }

# for table_name, df in tables.items():
#     if spark.catalog.tableExists(table_name):
#         print(f"Tabla {table_name} existe. Aplicando MERGE...")
#         delta_table = DeltaTable.forName(spark, table_name)
#         delta_table.alias("t").merge(
#             df.alias("s"),
#             "t.ID_REGISTRO_REL = s.ID_REGISTRO_REL"
#         ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
#     else:
#         print(f"Creando nueva tabla {table_name}...")
#         df.write.format("delta").mode("overwrite").saveAsTable(table_name)

In [0]:
table_name = "hta_sis.silver_hta"

if spark.catalog.tableExists(table_name):
    print(f"Tabla {table_name} existe. Aplicando MERGE...")
    delta_table = DeltaTable.forName(spark, table_name)
    delta_table.alias("t").merge(
        silver_hta.alias("s"),
        "t.ID_REGISTRO_REL = s.ID_REGISTRO_REL"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    print(f"Creando nueva tabla {table_name}...")
    silver_hta.write.format("delta").mode("overwrite").saveAsTable(table_name)